In [2]:
# pip install transformers torch

import torch
import torch.nn as nn
from transformers import BartTokenizer, BartModel, BartForConditionalGeneration

MODEL_NAME = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

D:\anaconda3\envs\web_lumplt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

# 1. SQuAD — Extractive QA

In [4]:
class BartQA(nn.Module):
    def __init__(self):
        super().__init__()
        self.bart = BartModel.from_pretrained(MODEL_NAME)
        self.qa_outputs = nn.Linear(self.bart.config.d_model, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=input_ids
        )

        hidden = outputs.last_hidden_state
        logits = self.qa_outputs(hidden)

        start_logits = logits[:, :, 0]
        end_logits = logits[:, :, 1]

        return start_logits, end_logits


model = BartQA()

question = "Where does John live?"
context = "John lives in Seattle."

text = question + " " + context

batch = tokenizer(text, return_tensors="pt")

start_logits, end_logits = model(
    batch["input_ids"],
    batch["attention_mask"]
)

start = start_logits.argmax(dim=-1).item()
end = end_logits.argmax(dim=-1).item()

answer_ids = batch["input_ids"][0, start:end+1]
print(tokenizer.decode(answer_ids))

 in Seattle


# 2. MNLI — Sentence Classification

In [4]:
class BartClassifier(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bart = BartModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Linear(self.bart.config.d_model, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=input_ids
        )

        final_hidden = outputs.last_hidden_state[:, -1, :]
        logits = self.classifier(final_hidden)

        return logits


model = BartClassifier(num_labels=3)

premise = "A dog is running in the park."
hypothesis = "An animal is outdoors."

batch = tokenizer(premise, hypothesis, return_tensors="pt")

logits = model(batch["input_ids"], batch["attention_mask"])

label_id = logits.argmax(dim=-1).item()

labels = {
    0: "entailment",
    1: "contradiction",
    2: "neutral"
}

print(labels[label_id])

contradiction


# 3. ELI5 — Long Answer Generation

In [5]:
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

question = "Why is the sky blue?"
documents = "Sunlight is scattered by molecules in the atmosphere."

input_text = "question: " + question + " context: " + documents

batch = tokenizer(input_text, return_tensors="pt", truncation=True)

output_ids = model.generate(
    batch["input_ids"],
    max_length=80,
    num_beams=4
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


question: Why is the sky blue? context: Sunlight is scattered by molecules in the atmosphere.


# 4. XSum — Abstractive Summarization

In [ ]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-xsum")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-xsum")

article = """
The patient was diagnosed with diabetes and received medicine at the hospital.
The doctor recommended continued treatment.
"""

batch = tokenizer(article, return_tensors="pt", truncation=True)

summary_ids = model.generate(
    batch["input_ids"],
    max_length=40,
    num_beams=4
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

# 5. ConvAI2 — Dialogue Response

In [ ]:
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

persona = "I like science. I enjoy helping people."
dialogue = "User: Hello, how are you?"

input_text = "persona: " + persona + " dialogue: " + dialogue + " response:"

batch = tokenizer(input_text, return_tensors="pt", truncation=True)

response_ids = model.generate(
    batch["input_ids"],
    max_length=50,
    num_beams=4
)

print(tokenizer.decode(response_ids[0], skip_special_tokens=True))

# 6. CNN/DailyMail — News Summarization

In [3]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

article = """
The company reported strong profits this quarter despite economic slowdown.
Executives said demand remained high across major markets.
"""

batch = tokenizer(article, return_tensors="pt", truncation=True)

summary_ids = model.generate(
    batch["input_ids"],
    max_length=60,
    min_length=10,
    num_beams=4
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

D:\anaconda3\envs\web_lumplt\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shiri\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. D

The company reported strong profits this quarter despite economic slowdown.Executives said demand remained high across major markets.
